# nb05 — Robustness of the selection (Supplementary Figure S16) and leave-cohort-out validation (S18)

**Plan:** `article_1_revision_implementation_plan_261029.md`, WP5 (robustness; Reviewer 1 comment 3,
Reviewer 2 comment 2 (iii)), WP7 (leave-cohort-out; Reviewer 1 comment 2) and WP9 §9.2 (new S16,
new S18).

**S16 question.** Does the ranking of the frozen selection rule depend on how the 87 metrics were
reduced or weighted? Each variant re-ranks every scenario pool with the same gates (marker
correlation, prediction) and different score bars; Kendall τ-b and the overlap of the Pareto
fronts are measured against the reference ranking.

**S18 question.** Does the approach the rule picks on part of the cohorts stay good on cohorts it
never saw? Four scenarios (RNA-seq only, FFPE only, FF only, all samples) × 5 folds: the rule is
applied to the 30 methods harmonized on the training cohorts, and the chosen method is ranked
among the 30 methods harmonized on the held-out test cohorts. Baseline: a random method, as a
distribution (Daniil 2026-11-08): uniform rank 1…n, Poisson-binomial hits, the decreases of every
test run; empirical p-values with Benjamini–Hochberg FDR. No baseline from the submitted article (Daniil, 2026-11-07: the revision never
compares its results with the pre-revision ones).

| Input | Content |
|---|---|
| `output/wp5/robustness_summary.tsv` | scenario × variant: Kendall τ-b, top-3 Jaccard, front Jaccard (r05) |
| `output/wp5/front_stability.tsv` | front member × variant: on the front or not (r05) |
| `output/wp7/v1_fold_results.tsv` | scenario × fold × role: rank on test, top-1 / top-3 hit, regrets (r22) |
| `output/wp7/v1_summary.tsv` | the same averaged per scenario (r22) |

**Inputs reflect the vetoes known on the run date.** r05 removes the vetoed approaches of
`output/wp4/vetoes.tsv` (round 1, 2026-11-04). Re-run r05 and this notebook after the round-2
visual check.

**Outputs** (`figures_svg/`): `figS16a_robustness_tau`, `figS16b_front_stability`,
`figS18a_v1_rank_on_test` (with the top-1 / top-3 hit tests since 2026-11-09), `figS18b_v1_regret`; tables `output/wp9/nb05_*.tsv`.

**Run:** `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=collagen_3_11 nb05_robustness_and_v1.ipynb` (from `article_1_revision/`).

Updated 2026-11-09 (Daniil): S18a half height, the hit-rate panel merged into S18a's titles; the decrease panel is now S18b.

Updated 2026-11-08 (figure review, plan `figure_review_corrections_plan_261108.md`): S16a faint bars = mean ± SD; S16b flat with thin bars; S18a bar plot with the random method as grey bars ± SD and failed / not-evaluable folds as text; S18b random share ± SD and exact Poisson-binomial p; S18c "decrease" (not "regret") as bars ± SD against every test run, Mann–Whitney + Monte Carlo p; every p with BH-FDR; every panel at its page size.

Updated 2026-11-07 (third review, gate G4): submitted-tree role removed; S18c with pairwise Mann–Whitney tests (BH FDR, stars). Earlier the same day: S16a as bars, S16b on the 22 approaches of the final selection, article palettes (`method_pal`, `strat_pal`) in S16 and S18.


## §0 Configuration

### §0a Imports

In [ ]:
# Imports. revision_lib is used only for plain-word names and paths (plan WP9).
import json
import textwrap
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu

import revision_lib as rl

PN = rl.PUBLICATION_NAMES  # every label comes from here (plan §1.1)

### §0b Paths and switches

| Switch | Default | Meaning |
|---|---|---|
| `TOP_K` | 3 | the hit criterion in the S18a titles next to top-1 (plan WP7: "transferred in x of 20 folds (top-3)") |
| `MIN_FRONT_STABILITY_SHOWN` | 0 | S16b lists every reference front member; raise it to show only unstable ones |
| `S18_REGRETS` | batch mixing, diagnosis retention, L, N | decreases drawn in S18b (0 = the chosen method is the best on the test part); internal column names `regret_*` |
| `MONTE_CARLO_DRAWS`, `MONTE_CARLO_SEED` | 100,000, 261108 | random draws behind the empirical p of S18a and S18b |

In [ ]:
# Paths and switches.
WP5_DIR = rl.OUTPUT_DIR / "wp5"
WP7_DIR = rl.OUTPUT_DIR / "wp7"
FIGURES_DIR = rl.REVISION_DIR / "figures_svg"
TABLES_DIR = rl.OUTPUT_DIR / "wp9"
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 3
MIN_FRONT_STABILITY_SHOWN = 0.0
S18_REGRETS = ["regret_batch_removal", "regret_biology_retention", "regret_L", "regret_N"]  # internal column names; shown as "decrease"
MONTE_CARLO_DRAWS = 100_000  # random draws behind every empirical p of S18 (Daniil 2026-11-08)
MONTE_CARLO_SEED = 261108

### §0c Plain-word names, palettes, plot settings

Variant names describe what changes against the reference (22-metric panel at |ρ| ≥ 0.8, headline
metric per bar). Scenario names are those of the selection rule; the leave-cohort-out scenarios
are strategies of the main benchmark.

In [ ]:
# Plain-word names of the robustness variants, the leave-cohort-out scenarios and roles.
VARIANT_NAMES = {
    "panel_cut0.1": "Panel built at |ρ| ≥ 0.9",
    "panel_cut0.3": "Panel built at |ρ| ≥ 0.7",
    "all_87_metrics": "Panel built from all 87 metrics",
    "axis_mean": "Bar = mean of its panel metrics",
    "group_balanced": "Bar = mean over metric groups",
    "local_runner_up": "Local bars: runner-up metrics",
    "bridging_biology": PN["bridging_biology"],
}
STATISTIC_NAMES = {
    "kendall_tau_b": "Kendall τ-b of the ranking",
    "top3_jaccard": "Jaccard index of the first three",
    "front_jaccard": "Jaccard index of the Pareto front",
}
V1_SCENARIO_NAMES = {
    "C": rl.STRATEGY_DISPLAY_NAMES["C_rnaseq_only"],
    "K": rl.STRATEGY_DISPLAY_NAMES["K_ffpe_only"],
    "J": rl.STRATEGY_DISPLAY_NAMES["J_ff_only"],
    "S0": rl.STRATEGY_DISPLAY_NAMES["S0_no_removal"],
}
RANDOM_NAME = "Random method"  # a distribution now (mean ± SD), not one expected value
ROLE_NAMES = {
    "chosen on the training part": PN["chosen_on_train"],
    "random method": RANDOM_NAME,
}
CHOSEN_COLOR, RANDOM_COLOR = "#1F3A5F", "#BBBBBB"  # the rule's choice dark, random light grey
ROLE_COLORS = {PN["chosen_on_train"]: CHOSEN_COLOR, RANDOM_NAME: RANDOM_COLOR}
# "Decrease", never "regret", in every figure (Daniil 2026-11-08); global / local named (D1).
DECREASE_NAMES = {
    "regret_batch_removal": "Decrease, batch mixing (global + local mean)",
    "regret_biology_retention": "Decrease, diagnosis retention (global + local mean)",
    "regret_L": "Decrease, marker correlation (local, within cohort)",
    "regret_N": "Decrease, cross-batch prediction",
}

# Article 1 palettes, copied (not imported) from the frozen figure notebook
# figures_for_article/Finally_assembled_figures_for_article.ipynb, cell 4. method_pal is
# that notebook's tab20 + tab20b palette (saturation + 0.15) over the 33 sorted methods,
# written out as hex so every revision figure uses exactly the submitted colours.
method_pal = {
    "01_raw": "#046cb4",
    "02_median_scaling": "#8bb3e8",
    "03_limma": "#ff7800",
    "04_sva": "#ffa852",
    "05_combat": "#14a014",
    "06_combat_seq": "#7cdf69",
    "07_pycombat": "#d60708",
    "08_inmoose_combatseq": "#ff7270",
    "09_ruv": "#864bbd",
    "10_mnn": "#b790d5",
    "11_harmony": "#8c4536",
    "12_scanorama": "#c48477",
    "13_fsmvn": "#e355b8",
    "14_qsmooth": "#f791bd",
    "15_fsqn_py": "#7f6c6c",
    "16_fsqn_r": "#c7a9a9",
    "17_quantile": "#bcbd06",
    "18_rank": "#dbdb6c",
    "19_tdm": "#00bccf",
    "20_shambhala": "#7cd5e5",
    "21_harmonizr": "#272979",
    "22_tmm": "#3a3ca3",
    "23_vst": "#4c50cf",
    "25_angel": "#7b7ede",
    "26_xpn": "#5d7927",
    "27_dwd": "#85a23a",
    "28_npn": "#adcf4c",
    "29_combat_ref": "#c7db7b",
    "31_ruv3prps": "#8c661c",
    "33_amdbnorm": "#bd971d",
    "34_arsyn": "#e7b02f",
    "36_explobatch": "#e7bf71",
    "38_harman": "#842925",
}
# covariate-free variants of the revision: the hue of the parent method, dashed line
method_pal["04_sva_nocov"] = method_pal["04_sva"]
method_pal["05_combat_nocov"] = method_pal["05_combat"]
strat_pal = {
    "S0_no_removal": "#a1c9f4",
    "A_confirmed_bad": "#ffb482",
    "B_extended_bad": "#8de5a1",
    "C_rnaseq_only": "#ff9f9b",
    "D_malignant_only": "#d0bbff",
    "E1_iterative_r1": "#debb9b",
    "E2_iterative_r2": "#fab0e4",
    "E3_iterative_r3": "#cfcfcf",
    "F_microarray_only": "#fffea3",
    "G_affymetrix_only": "#b9f2f0",
    "H_affymetrix_extended": "#bdb76b",
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only": "#7ecbc4",
    "K_ffpe_only": "#f2c27f",
}
imp_pal = {"strict": "#2979ae", "knn": "#982d22", "softimpute": "#713689"}
post_rm_pal = {False: "#CCCCCC", True: "#333333"}

# Scenario pools coloured by their strategy (strat_pal); the multi-platform pool spans 8
# strategies and takes a neutral dark grey.
# Pool names of r04 → plain words for the figures (no strategy codes such as "S0").
SCENARIO_PLAIN_NAMES = {
    "Multi-platform (RNA-seq + microarrays)": "RNA-seq + microarrays",
    "Microarrays only (not a tree leaf)": "Microarrays only",
    "No batch removal (S0 only)": "S0 · No removal",
}
SCENARIO_COLORS = {
    "FF only": strat_pal["J_ff_only"],
    "FFPE only": strat_pal["K_ffpe_only"],
    "RNA-seq only": strat_pal["C_rnaseq_only"],
    "RNA-seq + microarrays": "#4D4D4D",
    "Microarrays only": strat_pal["G_affymetrix_only"],
    "S0 · No removal": strat_pal["S0_no_removal"],
}


In [ ]:
# Plot settings (SVG + PDF + PNG 200 dpi) and helpers. Font sizes, slot_figsize and
# save_figure are redefined in Figma units by the next cell (Daniil 2026-11-09).
GLOBAL_FONT_SIZE = 7
SMALL_FONT_SIZE = 6  # smallest text on any panel (journal minimum)
plt.rcParams.update({
    "font.size": GLOBAL_FONT_SIZE,
    "axes.labelsize": GLOBAL_FONT_SIZE,
    "axes.titlesize": GLOBAL_FONT_SIZE + 1,
    "xtick.labelsize": GLOBAL_FONT_SIZE - 1,
    "ytick.labelsize": GLOBAL_FONT_SIZE - 1,
    "legend.fontsize": GLOBAL_FONT_SIZE - 1,
    "pdf.fonttype": "truetype",
    "svg.fonttype": "none",
    "figure.dpi": 100,
})
sns.set_style("ticks")


# Panel slots on the full figure page (D2, Daniil 2026-11-08). The page is the Figma frame
# "Figure 4" (node 761-28, 753 × 1150 px) = the 180 × 275 mm text area; every panel is drawn
# at its true size on that page, so GLOBAL_FONT_SIZE is the printed size in pt (≥ 6 pt).
# Copied from figures_svg/PLACEMENT.md (column "slot (mm)"); same dict in nb03–nb05.
PAGE_MM = (180, 275)
PANEL_SLOTS_MM = {
    "fig6B_eta_squared": (85, 50),
    "fig6_gradient_panel": (95, 50),
    "fig6CD_four_scores": (180, 110),
    "fig6F_decision_tree": (180, 110),
    "figS14a_marker_correlation_gate": (180, 85),
    "figS14b_prediction_gate": (180, 85),
    "figS14c_label_leakage": (180, 80),
    "figS16a_robustness_tau": (180, 140),
    "figS16b_front_stability": (180, 115),
    "figS17a_gradient_scores": (180, 125),
    "figS17b_gradient_slopes": (180, 55),
    "figS17c_observational_strategies": (180, 45),
    "figS17d_gradient_scatter": (180, 58),
    "figS17e_gradient_rank_trajectories": (180, 62),
    "figS17f_selected_methods_scores": (180, 105),
    "figS18a_v1_rank_on_test": (180, 50),  # half height (Daniil 2026-11-09)
    "figS18b_v1_regret": (180, 90),  # was S18c (S18b merged into S18a, 2026-11-09)
    "figS2_merged_distributions_all_cohorts": (180, 275),
}
MM_PER_INCH = 25.4


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    width_mm, height_mm = PANEL_SLOTS_MM[stem]
    return width_mm / MM_PER_INCH, height_mm / MM_PER_INCH


def save_figure(fig: plt.Figure, stem: str) -> None:
    # one file per panel: SVG + PDF + PNG (200 dpi)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


def save_table(table: pd.DataFrame, stem: str) -> None:
    # every plotted table is kept next to the figure inputs
    table.to_csv(TABLES_DIR / f"{stem}.tsv", sep="\t", index=False)
    print(f"saved output/wp9/{stem}.tsv ({len(table)} rows)")


# ── Statistical annotations (Daniil 2026-11-07, every comparison boxplot) ───────────
# Pairwise two-sided Mann–Whitney U tests, Benjamini–Hochberg FDR over every test of the
# figure, stars. ≤ 4 boxes: brackets with stars above the boxes; > 4 boxes: a heatmap of
# the FDR-adjusted p-values (pairwise_pvalue_heatmap) beside the boxplot.
def stars(q: float) -> str:
    # significance code of an FDR-adjusted p-value
    if not np.isfinite(q):
        return "n/a"
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns"


def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    # BH-adjusted p-values (monotone, capped at 1); NaN stays NaN
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p_values, np.nan)
    ok = np.isfinite(p_values)
    p = p_values[ok]
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(p)
    out[order] = np.minimum(ranked, 1.0)
    adjusted[ok] = out
    return adjusted


def pairwise_mannwhitney(data: pd.DataFrame, group_col: str, value_col: str, order: list,
                         panel: str = "") -> pd.DataFrame:
    # one row per pair of boxes: n, medians, U, raw p (FDR is added over the whole figure)
    rows = []
    for i, group_a in enumerate(order):
        for group_b in order[i + 1:]:
            a = data.loc[data[group_col] == group_a, value_col].dropna().to_numpy()
            b = data.loc[data[group_col] == group_b, value_col].dropna().to_numpy()
            if len(a) and len(b) and not (np.ptp(np.r_[a, b]) == 0):
                u, p = mannwhitneyu(a, b, alternative="two-sided")
            else:
                u, p = np.nan, np.nan
            rows.append({"panel": panel, "group_a": group_a, "group_b": group_b, "n_a": len(a), "n_b": len(b),
                         "median_a": np.median(a) if len(a) else np.nan,
                         "median_b": np.median(b) if len(b) else np.nan, "U": u, "p": p})
    return pd.DataFrame(rows)


def add_significance_bars(ax: plt.Axes, tests: pd.DataFrame, order: list, y_values: pd.Series) -> None:
    # brackets with stars above the boxes (≤ 4 boxes); stacked from the shortest pair up
    top = float(np.nanmax(y_values))
    span = top - float(np.nanmin(y_values)) or 1.0
    step = 0.09 * span
    level = top + 0.06 * span
    tests = tests.assign(width=[abs(order.index(a) - order.index(b)) for a, b in zip(tests["group_a"], tests["group_b"])])
    for row in tests.sort_values("width").itertuples():
        x1, x2 = order.index(row.group_a), order.index(row.group_b)
        ax.plot([x1, x1, x2, x2], [level, level + 0.02 * span, level + 0.02 * span, level], color="#333333", linewidth=0.9)
        ax.text((x1 + x2) / 2, level + 0.03 * span, stars(row.q_fdr), ha="center", va="bottom", fontsize=GLOBAL_FONT_SIZE - 1)
        level += step
    ax.set_ylim(top=level + 0.04 * span)


def pairwise_pvalue_heatmap(ax: plt.Axes, tests: pd.DataFrame, order: list, labels: list | None = None) -> None:
    # > 4 boxes: lower-triangle heatmap of −log10(FDR) with stars in the cells
    n = len(order)
    matrix = np.full((n, n), np.nan)
    marks = np.full((n, n), "", dtype=object)
    for row in tests.itertuples():
        i, j = sorted((order.index(row.group_a), order.index(row.group_b)), reverse=True)
        matrix[i, j] = -np.log10(max(row.q_fdr, 1e-300)) if np.isfinite(row.q_fdr) else np.nan
        marks[i, j] = stars(row.q_fdr)
    sns.heatmap(matrix, ax=ax, cmap="Reds", vmin=0, vmax=max(3, np.nanmax(matrix) if np.isfinite(matrix).any() else 3),
                annot=marks, fmt="", square=True, linewidths=0.4, linecolor="white",
                xticklabels=labels or order, yticklabels=labels or order,
                cbar_kws={"label": "−log10(FDR-adjusted p)", "shrink": 0.6}, annot_kws={"fontsize": SMALL_FONT_SIZE})
    ax.tick_params(axis="x", rotation=90)


In [ ]:
# Figma units (Daniil 2026-11-09, `daniil_review_261109.md`). Figma imports an SVG with
# 1 SVG pt = 4/3 px, and the figure frame (Figma node 1607-116664) is 756 × 1159 px for the
# 180 × 275 mm page. So every panel is drawn at (slot in Figma px) × 0.75 pt, and every text
# is 7.5 pt = 10 px in Figma. Legends and colour bars ("palettes") may stick out of the slot:
# only the plot area (axes with their tick labels and titles) is fitted to it.
import matplotlib.text as mtext
from matplotlib.transforms import Bbox

FIGMA_FRAME_PX = (756, 1159)
FIGMA_PAGE_MM = (180, 275)
FIGMA_PX_PER_PT = 4 / 3
FIGMA_PX_PER_MM = FIGMA_FRAME_PX[0] / FIGMA_PAGE_MM[0]  # 4.2 px per mm of the page
FIGMA_FONT_PX = 10
FIGMA_FONT_PT = FIGMA_FONT_PX / FIGMA_PX_PER_PT  # 7.5 pt
FIGMA_FIT_LOG = rl.OUTPUT_DIR / "wp14" / "figma_fit" / "nb05_robustness_and_v1.tsv"
FIGMA_FIT_LOG.parent.mkdir(parents=True, exist_ok=True)

GLOBAL_FONT_SIZE = FIGMA_FONT_PT
SMALL_FONT_SIZE = FIGMA_FONT_PT
plt.rcParams.update({key: FIGMA_FONT_PT for key in (
    "font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize", "ytick.labelsize",
    "legend.fontsize", "legend.title_fontsize", "figure.titlesize")})


def mm_figsize(width_mm: float, height_mm: float) -> tuple[float, float]:
    """Figure size in inches that lands in Figma at the slot size (page mm → Figma px)."""
    to_inches = FIGMA_PX_PER_MM / FIGMA_PX_PER_PT / 72
    return width_mm * to_inches, height_mm * to_inches


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    return mm_figsize(*PANEL_SLOTS_MM[stem])


def batch_display_name(batch: str) -> str:
    """Plain batch name: underscores → spaces; microarrays drop the last "Unknown" (library
    preparation is never known for a microarray), RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def is_palette_axis(ax: plt.Axes) -> bool:
    """Colour bars and legend-only axes are palettes: Daniil moves them by hand in Figma."""
    if hasattr(ax, "_colorbar") or getattr(ax, "_figma_palette", False):
        return True
    has_data = bool(ax.lines or ax.collections or ax.images or ax.patches or ax.texts)
    return not ax.axison and not has_data


def plot_extent_px(fig: plt.Figure) -> tuple[float, float]:
    """Width and height in Figma px of the plot area (axes, tick labels, titles, texts on
    the axes, suptitle), without legends and colour bars."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_tightbbox(renderer, bbox_extra_artists=list(ax.texts))
             for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax)]
    boxes += [t.get_window_extent(renderer) for t in fig.texts if t.get_text()]
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        boxes.append(fig._suptitle.get_window_extent(renderer))
    boxes = [b for b in boxes if b is not None and b.width > 0]
    # a legend-only figure has no plot area: measure the whole figure
    union = Bbox.union(boxes) if boxes else fig.bbox
    px_per_display = 72 / fig.dpi * FIGMA_PX_PER_PT
    return union.width * px_per_display, union.height * px_per_display


def set_all_fonts(fig: plt.Figure, size: float) -> None:
    """Every text of the figure at one size (10 px in Figma)."""
    for ax in fig.axes:
        ax.tick_params(labelsize=size)
    for text in fig.findobj(mtext.Text):
        if text.get_text():
            text.set_fontsize(size)


def fit_to_slot(fig: plt.Figure, width_mm: float, height_mm: float | None,
                n_iterations: int = 8) -> None:
    """Resize the figure until its plot area is width_mm wide (and height_mm high; None keeps
    the current aspect ratio of the plot area). Fonts keep their size, so a few redraws are
    needed; fixed-aspect plots (squares) only shrink to fit."""
    target_w = width_mm * FIGMA_PX_PER_MM
    for _ in range(n_iterations):
        plot_w, plot_h = plot_extent_px(fig)
        target_h = plot_h * target_w / plot_w if height_mm is None else height_mm * FIGMA_PX_PER_MM
        if abs(plot_w - target_w) < 1 and abs(plot_h - target_h) < 1:
            break
        fig_w, fig_h = fig.get_size_inches()
        fig.set_size_inches(max(fig_w * target_w / plot_w, 0.5), max(fig_h * target_h / plot_h, 0.5))
    plot_w, plot_h = plot_extent_px(fig)
    target_h = FIGMA_FRAME_PX[1] if height_mm is None else height_mm * FIGMA_PX_PER_MM
    shrink = min(1.0, target_w / plot_w, target_h / plot_h)
    if shrink < 0.995:  # fixed-aspect plots grew in the other direction: scale the whole panel
        fig.set_size_inches(*(fig.get_size_inches() * shrink))


def log_fit(stem: str, fig: plt.Figure, slot_mm: tuple | None) -> None:
    """One row per saved panel: slot and plot area in Figma px (read by r97_figma_check.py)."""
    plot_w, plot_h = plot_extent_px(fig)
    row = pd.DataFrame([{
        "stem": stem,
        "slot_w_px": None if slot_mm is None else round(slot_mm[0] * FIGMA_PX_PER_MM, 1),
        "slot_h_px": None if slot_mm is None or slot_mm[1] is None else round(slot_mm[1] * FIGMA_PX_PER_MM, 1),
        "plot_w_px": round(plot_w, 1), "plot_h_px": round(plot_h, 1),
    }])
    old = pd.read_csv(FIGMA_FIT_LOG, sep="\t") if FIGMA_FIT_LOG.exists() else row.iloc[:0]
    pd.concat([old[old["stem"] != stem], row]).to_csv(FIGMA_FIT_LOG, sep="\t", index=False)


def save_figure(fig: plt.Figure, stem: str, slot_mm: tuple | None = None,
                keep_fonts: bool = False) -> None:
    """Fonts → 10 Figma px (unless keep_fonts: large clustermaps), plot area → slot (default:
    PANEL_SLOTS_MM[stem] when the notebook has it), then SVG + PDF + PNG (200 dpi)."""
    if slot_mm is None and stem in globals().get("PANEL_SLOTS_MM", {}):
        slot_mm = PANEL_SLOTS_MM[stem]
    if not keep_fonts:
        set_all_fonts(fig, FIGMA_FONT_PT)
    if slot_mm is not None:
        fit_to_slot(fig, *slot_mm)
    log_fit(stem, fig, slot_mm)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


## §1 Supplementary Figure S16a: ranking agreement across variants

Expected: 6 scenario pools × 7 variants = 42 rows. τ-b is computed over the whole pool (153–1,254
approaches); the two Jaccard indices look only at the top of the ranking. Gate G4 review (Daniil,
2026-11-07): drawn as bars so that "whole ranking stable, top unstable" is visible at once. Since
2026-11-08: bar = mean over the 6 pools in a faint colour (so the pool dots stay readable), error
bar = SD over the pools, dots = pools coloured by strategy.

In [ ]:
# Load the robustness summary and draw one bar panel per statistic.
robustness_summary = pd.read_csv(WP5_DIR / "robustness_summary.tsv", sep="\t")
robustness_summary["scenario"] = robustness_summary["scenario"].replace(SCENARIO_PLAIN_NAMES)
robustness_summary["variant_name"] = robustness_summary["variant"].map(VARIANT_NAMES)
assert robustness_summary["variant_name"].notna().all(), "a variant without a plain-word name"
assert robustness_summary["scenario"].isin(SCENARIO_COLORS).all(), "a scenario without a colour"
print(f"rows: {len(robustness_summary)}; rule MD5: {robustness_summary['rule_md5'].unique()}")
save_table(robustness_summary, "nb05_robustness_summary")

variant_order = [VARIANT_NAMES[v] for v in VARIANT_NAMES if v in set(robustness_summary["variant"])]
STATISTIC_COLORS = {"kendall_tau_b": "#1F3A5F", "top3_jaccard": "#C0504D", "front_jaccard": "#D9A441"}
fig, axes = plt.subplots(1, 3, figsize=slot_figsize("figS16a_robustness_tau"), sharey=True)
x_positions = np.arange(len(variant_order))
for ax, statistic in zip(axes, STATISTIC_NAMES):
    # bar = mean over the six pools, faint so the pool dots stay readable; error bar = SD
    per_variant = robustness_summary.groupby("variant_name")[statistic].agg(["mean", "std"]).loc[variant_order]
    ax.bar(x_positions, per_variant["mean"], color=STATISTIC_COLORS[statistic], alpha=0.25, width=0.65,
           yerr=per_variant["std"], error_kw={"elinewidth": 0.9, "capsize": 2.5, "ecolor": "#333333"})
    for offset, (scenario, color) in zip(np.linspace(-0.22, 0.22, len(SCENARIO_COLORS)), SCENARIO_COLORS.items()):
        values = robustness_summary[robustness_summary["scenario"] == scenario].set_index("variant_name")[statistic].reindex(variant_order)
        ax.scatter(x_positions + offset, values, s=12, color=color, edgecolor="#333333", linewidth=0.4, zorder=3)
    for x, value in zip(x_positions, per_variant["mean"]):
        ax.text(x, 0.02, f"{value:.2f}", ha="center", va="bottom", fontsize=SMALL_FONT_SIZE, color="#333333")  # mean at the bar base, clear of the dots
    ax.set_title(STATISTIC_NAMES[statistic], fontsize=GLOBAL_FONT_SIZE)
    ax.set_xticks(x_positions, variant_order, rotation=50, ha="right", fontsize=SMALL_FONT_SIZE)
    ax.set_ylim(0, 1.08)
    ax.grid(axis="y", color="#EEEEEE", linewidth=0.6)
    sns.despine(ax=ax)
axes[0].set_ylabel("Agreement with the reference\n(1 = identical)")
handles = [mlines.Line2D([], [], marker="o", linestyle="none", markerfacecolor=c, markeredgecolor="#333333", label=s)
           for s, c in SCENARIO_COLORS.items()]
handles.append(mlines.Line2D([], [], color="#333333", label="Bar = mean over the 6 pools, error bar = SD"))
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "figS16a_robustness_tau")
plt.show()
print(robustness_summary.groupby("variant_name")[list(STATISTIC_NAMES)].agg(["mean", "std"]).round(2).to_string())

## §2 Supplementary Figure S16b: stability of each selected approach

For every approach of the final selection (rule front, accepted at the visual check, gate G2b closed
2026-11-07): the share of the 8 rankings (reference + 7 variants) in which it stays on the front.
Bar colour = harmonization method (`method_pal`).

In [ ]:
# Front stability of the 22 approaches of the final selection.
front_stability = pd.read_csv(WP5_DIR / "front_stability.tsv", sep="\t")
front_stability["scenario"] = front_stability["scenario"].replace(SCENARIO_PLAIN_NAMES)
selected_mask = front_stability["selected_after_visual_check"].fillna(False).astype(bool)
reference_front = front_stability[selected_mask & (front_stability["front_stability"] >= MIN_FRONT_STABILITY_SHOWN)].copy()
assert len(reference_front) == 22, "the final selection has 22 approaches (gate G2b)"
reference_front["approach_label"] = (
    reference_front["method_display"] + ", "
    + reference_front["strat"].map(rl.STRATEGY_DISPLAY_NAMES) + ", "
    + reference_front["imp"].map({"strict": "no imputation", "knn": "KNN", "softimpute": "softImpute"})
    + np.where(reference_front["post_rm"], ", post-removal", "")
)
reference_front = reference_front.sort_values(["scenario", "front_stability"], ascending=[True, False])
save_table(reference_front, "nb05_front_stability_reference_members")
print(reference_front.groupby("scenario")["front_stability"].agg(["count", "median", "min"]).round(2).to_string())

scenarios = reference_front["scenario"].unique().tolist()
heights = [max(1, (reference_front["scenario"] == s).sum()) for s in scenarios]
fig, axes = plt.subplots(len(scenarios), 1, figsize=slot_figsize("figS16b_front_stability"),  # flat (Daniil 2026-11-08)
                         gridspec_kw={"height_ratios": heights}, sharex=True)
for ax, scenario in zip(axes, scenarios):
    rows = reference_front[reference_front["scenario"] == scenario]
    ax.barh(rows["approach_label"], rows["front_stability"], color=rows["method"].map(method_pal), edgecolor="#333333", linewidth=0.4, height=0.35)
    for y, value in enumerate(rows["front_stability"]):
        ax.text(value + 0.01, y, f"{value:.2f}", va="center", fontsize=SMALL_FONT_SIZE)
    ax.invert_yaxis()
    ax.set_title(scenario, loc="left", fontsize=SMALL_FONT_SIZE)
    ax.tick_params(axis="y", labelsize=SMALL_FONT_SIZE)
    ax.set_xlim(0, 1.1)
    ax.grid(axis="x", color="#EEEEEE", linewidth=0.6)
    sns.despine(ax=ax)
axes[-1].set_xlabel("Share of the 8 rankings with the approach on the Pareto front")
fig.tight_layout()
save_figure(fig, "figS16b_front_stability")
plt.show()

## §3 Leave-cohort-out validation: load the fold results

Expected: 20 folds (4 scenarios × 5). Four test folds are one large single-batch, single-diagnosis
cohort (RNA-seq only 2, FFPE only 2): batch metrics and the prediction gate cannot be evaluated
there (plan caveat C4). When the chosen method fails on the test part (ComBat with the diagnosis
covariate cannot fit a test fold where diagnosis and batch coincide) it has no rank; such folds
count as "no hit". In S18a both kinds of fold carry a vertical note instead of a bar.

**Random baseline (Daniil 2026-11-08).** A method drawn at random among the n test runs of a fold
has a rank uniform on 1…n: mean (n + 1)/2, SD √((n² − 1)/12), P(random rank ≤ r) = r / n exactly,
a top-k hit with probability min(k, n)/n; its decrease is that of one of the n test runs. Tables:
`nb05_v1_random_baseline.tsv` (per fold), `nb05_v1_random_decreases.tsv` (per test run; their mean
equals the expected value r22 stored, checked).

In [ ]:
# Load the fold results and the per-scenario summary.
fold_results = pd.read_csv(WP7_DIR / "v1_fold_results.tsv", sep="\t")
for column in ["test_evaluable", "top1_hit", "top3_hit"]:
    # booleans for the chosen / tree rows, expected shares (e.g. 0.033) for the random row
    fold_results[column] = pd.to_numeric(fold_results[column].replace({"True": 1.0, "False": 0.0, True: 1.0, False: 0.0}))
fold_results["role_name"] = fold_results["role"].map(ROLE_NAMES)
fold_results["scenario_name"] = fold_results["scenario"].map(V1_SCENARIO_NAMES)
fold_results["failed_on_test"] = fold_results["rank_on_test"].isna() & (fold_results["role"] != "random method")
chosen = fold_results[fold_results["role"] == "chosen on the training part"].copy()
print(f"folds: {len(chosen)}; evaluable test folds: {int(chosen['test_evaluable'].sum())}")
print(f"chosen method failed on the test part: {int(chosen['failed_on_test'].sum())} folds")
print(chosen[["scenario_name", "fold", "method_display", "rank_on_test", "top1_hit", "top3_hit", "test_evaluable"]].to_string(index=False))
save_table(fold_results, "nb05_v1_fold_results")
assert len(chosen) == 20

# ── Random baseline as a distribution (Daniil 2026-11-08) ─────────────────────────
# A method drawn at random among the n test runs of a fold has a rank uniform on 1…n:
# mean (n + 1) / 2, SD √((n² − 1) / 12), P(random rank ≤ r) = r / n exactly, and a top-k hit
# with probability min(k, n) / n. Its decrease (best test-run value − value of the run) is
# the decrease of one of the n test runs, so the random distribution is the list of the
# decreases of every test run of the fold (v1_scores.parquet).
n_runs = chosen["n_test_runs"].astype(float)
random_rank_baseline = chosen[["scenario", "scenario_name", "fold", "method", "test_evaluable", "failed_on_test", "rank_on_test"]].assign(
    n_test_runs=n_runs.astype(int),
    expected_rank=(n_runs + 1) / 2,
    sd_rank=np.sqrt((n_runs ** 2 - 1) / 12),
    p_rank=chosen["rank_on_test"] / n_runs,  # exact P(random rank ≤ observed rank)
    p_top1=np.minimum(1, n_runs) / n_runs,
    **{f"p_top{TOP_K}": np.minimum(TOP_K, n_runs) / n_runs},
)
save_table(random_rank_baseline, "nb05_v1_random_baseline")

DECREASE_SCORES = {regret: regret.removeprefix("regret_") for regret in S18_REGRETS}  # decrease column → score
v1_scores = pd.read_parquet(WP7_DIR / "v1_scores.parquet")
test_runs = v1_scores[v1_scores["part"] == "test"]
random_decrease_rows = []
for (scenario, fold), runs in test_runs.groupby(["scenario", "fold"]):
    for regret, score in DECREASE_SCORES.items():
        values = runs[score].dropna()
        decreases = values.max() - values
        random_decrease_rows += [{"scenario": scenario, "fold": int(fold), "decrease_of": regret, "method": m, "decrease": d}
                                 for m, d in zip(runs.loc[values.index, "method"], decreases)]
random_decreases = pd.DataFrame(random_decrease_rows)
save_table(random_decreases, "nb05_v1_random_decreases")
# the mean of these per-run decreases is the expected value r22 stored for the random role
check = random_decreases.groupby(["scenario", "fold", "decrease_of"])["decrease"].mean()
stored = fold_results[fold_results["role"] == "random method"].melt(
    id_vars=["scenario", "fold"], value_vars=S18_REGRETS, var_name="decrease_of", value_name="stored"
).set_index(["scenario", "fold", "decrease_of"])["stored"]
assert np.allclose(check.reindex(stored.index), stored, equal_nan=True), "per-run decreases differ from r22"
print(f"random baseline: {len(random_rank_baseline)} folds; {len(random_decreases)} per-run decreases")


## §4 Supplementary Figure S18a: rank on the test part

Per fold: coloured bar = rank of the chosen method among the methods harmonized on the test cohorts
(1 = best by the rule order; colour = method, `method_pal`), grey bar = random method, mean ± SD of
the uniform rank. Per scenario: Monte Carlo of 100,000 draws of one random rank per fold, empirical
p = P(Σ random ranks ≤ Σ observed ranks), Benjamini–Hochberg FDR over the 4 scenarios (in the panel
title). Folds whose test part is not evaluable or where the chosen method failed have no bar and
are not tested. Table `output/wp9/nb05_v1_rank_tests.tsv`.


In [ ]:
# Rank on test per fold: chosen method (bar, method colour) vs random method (grey bar, mean ± SD).
# Statistics per scenario: Monte Carlo of the sum of one random rank per fold (MONTE_CARLO_DRAWS,
# fixed seed); empirical p = P(Σ random ranks ≤ Σ observed ranks); BH-FDR over the scenarios.
# Folds without a bar (test part not evaluable, chosen method failed) carry a vertical note and
# are left out of the test.
rng = np.random.default_rng(MONTE_CARLO_SEED)


def monte_carlo_rank_p(ranks: np.ndarray, n_runs: np.ndarray) -> tuple[float, float]:
    """Empirical P(sum of one uniform random rank per fold ≤ observed sum) and the random mean sum."""
    draws = sum(rng.integers(1, n + 1, size=MONTE_CARLO_DRAWS) for n in n_runs.astype(int))
    observed = ranks.sum()
    return (np.sum(draws <= observed) + 1) / (MONTE_CARLO_DRAWS + 1), float(draws.mean())


scenario_order = list(V1_SCENARIO_NAMES.values())
drawn = random_rank_baseline["test_evaluable"].astype(bool) & ~random_rank_baseline["failed_on_test"]
rank_tests = []
for scenario in scenario_order:
    rows = random_rank_baseline[(random_rank_baseline["scenario_name"] == scenario) & drawn]
    p, random_sum = monte_carlo_rank_p(rows["rank_on_test"].to_numpy(), rows["n_test_runs"].to_numpy())
    rank_tests.append({"scenario_name": scenario, "n_folds_tested": len(rows), "sum_observed_rank": rows["rank_on_test"].sum(),
                       "sum_random_rank_mean": random_sum, "p_empirical": p})
rank_tests = pd.DataFrame(rank_tests)
rank_tests["q_fdr"] = benjamini_hochberg(rank_tests["p_empirical"].to_numpy())
rank_tests["stars"] = rank_tests["q_fdr"].map(stars)
save_table(rank_tests, "nb05_v1_rank_tests")
print(rank_tests.round(4).to_string(index=False))
# S18a is drawn in §5, after the hit tests (its panel titles carry both).


## §5 Supplementary Figure S18a: top-1 / top-3 hit tests and the drawing

S18b (hit rates) was merged into S18a on 2026-11-09 (Daniil: both panels answer the same question;
the hits are the ranks of S18a with a threshold). The tests stay: share of the 20 folds in which the
chosen method is the best (top-1) or among the first three (top-3) on the test part (failed on the
test part = no hit); the number of random hits is Poisson-binomial with per-fold probabilities
min(k, n)/n, so the exact p = P(random hits ≥ observed hits); Benjamini–Hochberg FDR over the
5 groups × 2 thresholds. The S18a panel titles carry the per-scenario counts and FDR. Table
`output/wp9/nb05_v1_hit_tests.tsv`.


In [ ]:
# Hit rates per scenario: observed share (bar) vs random method (grey bar, expected share ± SD).
# Over all 20 folds (failed on test = no hit). The number of random hits is Poisson-binomial with
# per-fold probabilities min(k, n) / n: exact empirical p = P(random hits ≥ observed hits);
# BH-FDR over the 5 groups × 2 panels; stars by the standard rule.
def poisson_binomial_pmf(probabilities: np.ndarray) -> np.ndarray:
    """Exact distribution of the number of successes of independent trials (convolution)."""
    pmf = np.array([1.0])
    for p in probabilities:
        pmf = np.convolve(pmf, [1 - p, p])
    return pmf


hit_rows = []
groups = [(name, chosen["scenario_name"] == name) for name in scenario_order] + [("All scenarios", chosen["scenario_name"].notna())]
for k, column in [(1, "top1_hit"), (TOP_K, f"top{TOP_K}_hit")]:
    for name, mask in groups:
        part = random_rank_baseline[mask.to_numpy()]
        observed = int(chosen.loc[mask, column].fillna(0).sum())
        probabilities = np.minimum(k, part["n_test_runs"]) / part["n_test_runs"]
        pmf = poisson_binomial_pmf(probabilities.to_numpy())
        hit_rows.append({"top_k": k, "scenario_name": name, "n_folds": len(part), "observed_hits": observed,
                         "observed_share": observed / len(part), "random_share": probabilities.mean(),
                         "random_share_sd": np.sqrt(np.sum(probabilities * (1 - probabilities))) / len(part),
                         "p_exact": float(pmf[observed:].sum())})
hit_tests = pd.DataFrame(hit_rows)
hit_tests["q_fdr"] = benjamini_hochberg(hit_tests["p_exact"].to_numpy())
hit_tests["stars"] = hit_tests["q_fdr"].map(stars)
save_table(hit_tests, "nb05_v1_hit_tests")
print(hit_tests.round(4).to_string(index=False))


In [ ]:
# Draw S18a: per-fold ranks (chosen vs random) with the rank test and the top-1 / top-3
# hit tests in the panel titles (S18b merged here, Daniil 2026-11-09).
fig, axes = plt.subplots(1, len(scenario_order), figsize=slot_figsize("figS18a_v1_rank_on_test"), sharey=True)
BAR_WIDTH = 0.38
for ax, scenario in zip(axes, scenario_order):
    rows = random_rank_baseline[random_rank_baseline["scenario_name"] == scenario]
    top = rows["n_test_runs"].max()
    for row in rows.itertuples():
        x = row.fold
        ax.bar(x + BAR_WIDTH / 2, row.expected_rank, width=BAR_WIDTH, color=RANDOM_COLOR, yerr=row.sd_rank,
               error_kw={"elinewidth": 0.8, "capsize": 2, "ecolor": "#555555"})
        if row.failed_on_test or not row.test_evaluable:
            note = "method failed\non the test part" if row.failed_on_test else "test part\nnot evaluable"
            ax.text(x - BAR_WIDTH / 2, 0.5, note, rotation=90, ha="center", va="bottom", fontsize=SMALL_FONT_SIZE, color="#555555")
            continue
        ax.bar(x - BAR_WIDTH / 2, row.rank_on_test, width=BAR_WIDTH, color=method_pal[row.method], edgecolor="#333333", linewidth=0.5)
    test = rank_tests.set_index("scenario_name").loc[scenario]
    hits = hit_tests.set_index(["top_k", "scenario_name"])
    top1, top3 = hits.loc[(1, scenario)], hits.loc[(TOP_K, scenario)]
    # merged S18b (Daniil 2026-11-09): the hit counts are the same ranks with a threshold
    ax.set_title(f"{scenario}\nrank: p {test['p_empirical']:.2g}, FDR {test['q_fdr']:.2g} {test['stars']}\n"
                 f"top-1: {top1.observed_hits}/{top1.n_folds}, FDR {top1.q_fdr:.2g} {top1.stars}\n"
                 f"top-{TOP_K}: {top3.observed_hits}/{top3.n_folds}, FDR {top3.q_fdr:.2g} {top3.stars}",
                 fontsize=SMALL_FONT_SIZE)
    ax.axhline(TOP_K + 0.5, color="#999999", linewidth=0.8, linestyle="--")
    ax.set_xticks(range(1, 6))
    ax.set_ylim(0, top + 1)
    ax.grid(axis="y", color="#EEEEEE", linewidth=0.6)
    ax.set_xlabel("Test fold")
    sns.despine(ax=ax)
axes[0].set_ylabel("Rank on the test part\n(1 = best; lower bar = better)")
shown_methods = sorted(chosen.loc[drawn.to_numpy(), "method"].dropna().unique())
handles = [mpatches.Patch(color=method_pal[m], label=f"{PN['chosen_on_train']}: {rl.method_display_name(m)}") for m in shown_methods]
handles += [mpatches.Patch(color=RANDOM_COLOR, label="Random method, mean ± SD (uniform rank 1…n)"),
            mlines.Line2D([], [], color="#999999", linestyle="--", label=f"Top-{TOP_K} limit"),
            mlines.Line2D([], [], linestyle="none", label="Titles: rank, Monte Carlo p; top-1 / top-3 hits, exact Poisson-binomial p; BH FDR")]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=3, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "figS18a_v1_rank_on_test")
plt.show()


## §6 Supplementary Figure S18b: decrease (S18c until 2026-11-09)

Decrease (called "regret" in the internal tables; Daniil 2026-11-08: "decrease" is the technical
word) = best value among the test-part methods minus the value of the method; 0 = it is the best
on that score. Panel names say which scores are global + local means (D1). Chosen: one value per
evaluable fold where the chosen method ran (dots); random: the decreases of every test run of the
same folds. Bars = mean ± SD. Tests per panel: two-sided Mann–Whitney (chosen vs random runs) and a
Monte Carlo empirical p of the mean decrease (100,000 draws of one random run per fold);
Benjamini–Hochberg FDR over the four panels for each; stars from the Mann–Whitney FDR.
Test table: `output/wp9/nb05_v1_regret_tests.tsv` (file name internal).


In [ ]:
# Decrease (formerly "regret"; Daniil 2026-11-08: "decrease" is the technical word) on four
# scores: chosen method vs random method, bars = mean ± SD.
# Chosen: one value per evaluable fold where the chosen method ran. Random: the decreases of
# every test run of the same folds (a distribution, not one expected value per fold).
# Tests per panel: two-sided Mann–Whitney (chosen folds vs random runs) and a Monte Carlo
# empirical p of the mean (one random run per fold, MONTE_CARLO_DRAWS draws, fixed seed:
# P(mean random decrease ≤ observed mean)); BH-FDR over the four panels for each; stars from
# the Mann–Whitney FDR.
evaluable_folds = chosen.loc[chosen["test_evaluable"].astype(bool), ["scenario", "fold"]]
decrease_rows, decrease_tests = [], []
for regret in S18_REGRETS:
    observed = chosen.merge(evaluable_folds, on=["scenario", "fold"])[["scenario", "fold", regret]].dropna()
    pool = random_decreases[random_decreases["decrease_of"] == regret].merge(observed[["scenario", "fold"]], on=["scenario", "fold"])
    u, p_mw = mannwhitneyu(observed[regret], pool["decrease"], alternative="two-sided")
    per_fold = [group["decrease"].to_numpy() for _, group in pool.groupby(["scenario", "fold"])]
    draws = np.mean([rng.choice(values, size=MONTE_CARLO_DRAWS) for values in per_fold], axis=0)
    p_mc = (np.sum(draws <= observed[regret].mean()) + 1) / (MONTE_CARLO_DRAWS + 1)
    decrease_tests.append({"panel": regret, "panel_name": DECREASE_NAMES[regret], "n_a": len(observed), "n_b": len(pool),
                           "mean_a": observed[regret].mean(), "sd_a": observed[regret].std(),
                           "mean_b": pool["decrease"].mean(), "sd_b": pool["decrease"].std(),
                           "median_a": observed[regret].median(), "median_b": pool["decrease"].median(),
                           "U": u, "p": p_mw, "p_monte_carlo": p_mc})
    decrease_rows += [{"panel": regret, "role_name": PN["chosen_on_train"], "value": v} for v in observed[regret]]
    decrease_rows += [{"panel": regret, "role_name": RANDOM_NAME, "value": v} for v in pool["decrease"]]
decrease_tests = pd.DataFrame(decrease_tests)
decrease_tests["q_fdr"] = benjamini_hochberg(decrease_tests["p"].to_numpy())  # Mann–Whitney FDR (r45 key v1_regret_fdr__*)
decrease_tests["q_fdr_monte_carlo"] = benjamini_hochberg(decrease_tests["p_monte_carlo"].to_numpy())
decrease_tests["stars"] = decrease_tests["q_fdr"].map(stars)
save_table(decrease_tests, "nb05_v1_regret_tests")  # file name internal, kept for r45
decrease_long = pd.DataFrame(decrease_rows)
save_table(decrease_long, "nb05_v1_regret_long")

fig, axes = plt.subplots(1, len(S18_REGRETS), figsize=slot_figsize("figS18b_v1_regret"), sharey=False)
for ax, regret in zip(axes, S18_REGRETS):
    test = decrease_tests.set_index("panel").loc[regret]
    means, sds = [test["mean_a"], test["mean_b"]], [test["sd_a"], test["sd_b"]]
    ax.bar([0, 1], means, yerr=sds, width=0.6, color=[CHOSEN_COLOR, RANDOM_COLOR], alpha=0.85,
           error_kw={"elinewidth": 0.8, "capsize": 3, "ecolor": "#333333"})
    folds = decrease_long[(decrease_long["panel"] == regret) & (decrease_long["role_name"] == PN["chosen_on_train"])]["value"]
    ax.scatter(np.zeros(len(folds)) + rng.uniform(-0.12, 0.12, len(folds)), folds, s=6, color="#222222", zorder=3)
    top = max(m + s for m, s in zip(means, sds))
    ax.plot([0, 0, 1, 1], [top * 1.08, top * 1.12, top * 1.12, top * 1.08], color="#333333", linewidth=0.8)
    ax.text(0.5, top * 1.14, test["stars"], ha="center", va="bottom", fontsize=SMALL_FONT_SIZE)
    ax.set_ylim(0, top * 1.3)
    ax.set_title(textwrap.fill(DECREASE_NAMES[regret], 24), fontsize=SMALL_FONT_SIZE)
    ax.set_xticks([0, 1], ["Chosen on the\ntraining part", "Random\nmethod"], fontsize=SMALL_FONT_SIZE)
    ax.set_ylabel("Decrease vs the best approach\non the test part (0 = best)" if ax is axes[0] else "")
    ax.grid(axis="y", color="#EEEEEE", linewidth=0.6)
    sns.despine(ax=ax)
fig.text(0.5, -0.04, "Bars: mean ± SD (chosen: one value per evaluable fold, dots; random: every test run of the same folds).\n"
         "Two-sided Mann–Whitney U test, Benjamini–Hochberg FDR over the four panels; * < 0.05, ** < 0.01, *** < 0.001, ns.",
         ha="center", va="top", fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "figS18b_v1_regret")
plt.show()
print(decrease_tests[["panel_name", "n_a", "n_b", "mean_a", "mean_b", "p", "q_fdr", "p_monte_carlo", "q_fdr_monte_carlo", "stars"]].round(4).to_string(index=False))


## §7 Notes for the text (gate G4 answered 2026-11-07)

- Leave-cohort-out is reported as "transferred in x of 20 folds (top-3)"; the 4 folds where the
  chosen method failed on the test part count as no hit and are named (G4.4, option a).
- FFPE only: the chosen method is never in the test top-3 (0 / 5; 3 of the 5 chosen methods are
  ComBat / limma runs that cannot run on the test cohorts). The text says so plainly.
- S0 · No removal: Angel is chosen in all 5 folds and is first on the test part in 3 of them.
- The decrease test (2026-11-08) compares the fold values of the chosen method with the decreases
  of every test run of the same evaluable folds (unpaired Mann–Whitney) and adds a Monte Carlo
  empirical p of the mean; the random baseline is a distribution everywhere in S18, with an
  empirical p and BH-FDR (Daniil's figure rule 2026-11-08).
